
# Fig. 5b — Common cell types: high-attention genes vs canonical markers

这个 notebook 用于回答一个非常直观的机制问题：

> 当 Annotation Decoder 在真实自然语言 annotation 中生成某个常见 cell type 的 identity span 时，
> 它对 encoder genes 的 cross-attention 是否集中到公认的 canonical marker genes？

固定分析 6 类：

- T cell
- B cell
- NK cell（数据中 exact label = `natural killer cell`）
- classical monocyte
- plasma cell
- neutrophil

### 分析流程

```text
VAL cells
  ↓
只选择 6 个预先指定的 cell types
  ↓
teacher-force 原始 natural_language_annotation
  ↓
精确定位真实 cell-type text span
  ↓
最后一层 Annotation Decoder → Encoder cross-attention
  ↓
heads × cell-type query tokens 平均
  ↓
每个 cell 内转换为 gene attention percentile
  ↓
同一 cell type 内按 gene 聚合
  ↓
按 median attention percentile 排名
  ↓
Top-N genes 与预先固定的 external canonical marker list 对照
```

### 关键原则

- **不使用 TRAIN-derived markers 来定义 canonical marker。**
- canonical marker list 在查看模型 gene-attention 排名之前固定。
- 不做 fuzzy/synonym span matching。
- 机制分析使用 **unmasked VAL transcriptomes (`mlm_probability=0`)**。
- Top gene 排名要求 gene 在该 cell type 至少一定比例的 analyzed cells 中进入 encoder，避免极少数细胞中的偶然高 attention。


In [ ]:





from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'sckite').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'sckite').is_dir():
    raise FileNotFoundError('Run this notebook from inside the scKITE repository.')
STAGE2_YAML = REPO_ROOT / 'sckite' / 'stage2' / 'config.yaml'
STAGE2_CKPT = REPO_ROOT / 'checkpoints' / 'stage2' / 'best.pt'


PRETRAIN_VAL = None

OUT_DIR = REPO_ROOT / 'outputs' / 'biological_interpretation' / 'canonical_marker_attention'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CELL_ID_FIELD = "cell_id"
CELL_TYPE_FIELD = "cell_type"
ANNOTATION_FIELD = "natural_language_annotation"
GENES_FIELD = "genes"
EXPRESSIONS_FIELD = "expressions"

ATTENTION_BATCH_SIZE = 20
MAX_CELLS_PER_TYPE = None

MIN_GENE_CELLS = 5
MIN_GENE_PRESENCE_FRACTION = 0.20

TOP_N_GENES = 15
TOP_K_FOR_OVERLAP = 15


ANALYSIS_MLM_PROBABILITY = 0.0



ARIAL_TTF = None

TITLE_SIZE = 9
LABEL_SIZE = 8
TICK_SIZE = 8

FORCE_RESCAN = False
FORCE_RERUN_ATTENTION = False

SEED = 2026

print("OUT_DIR:", OUT_DIR)


In [ ]:





import os
import sys
import re
import json
import random
import warnings
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import yaml
from tqdm.auto import tqdm

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

try:
    from streaming import StreamingDataset
except Exception as exc:
    raise ImportError("无法导入 streaming.StreamingDataset。") from exc

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sckite.stage2.tokenizer import GlobalGeneTextTokenizer
from sckite.stage2.data import Stage2Collator
from sckite.stage2.model import ScKITEStage2Model

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


def configure_arial(arial_ttf=None):

    if arial_ttf is not None:
        arial_ttf = Path(arial_ttf).expanduser()
        if not arial_ttf.exists():
            raise FileNotFoundError(f"ARIAL_TTF 不存在: {arial_ttf}")
        fm.fontManager.addfont(str(arial_ttf))

    try:
        arial_path = fm.findfont("Arial", fallback_to_default=False)
        print("Arial found:", arial_path)
        font_family = "Arial"
    except Exception:
        font_family = "sans-serif"
        warnings.warn(
            "当前 Python/Matplotlib 环境未找到 Arial。"
            "图仍可生成，但正式投稿前请安装 Arial 或在 ARIAL_TTF 中提供 Arial.ttf 路径。"
        )

    mpl.rcParams.update({
        "font.family": font_family,
        "axes.unicode_minus": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })

    return font_family


PLOT_FONT = configure_arial(ARIAL_TTF)
print("plot font:", PLOT_FONT)


In [ ]:





with open(STAGE2_YAML, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

if PRETRAIN_VAL is None:
    PRETRAIN_VAL = Path(cfg["paths"]["val_local"])
else:
    PRETRAIN_VAL = Path(PRETRAIN_VAL)

gv_cfg = cfg["global_vocab"]
data_cfg = cfg["data"]
decoder_cfg = cfg["decoder"]

tokenizer = GlobalGeneTextTokenizer.from_files(
    text_tokenizer_path=gv_cfg["text_tokenizer_path"],
    global_vocab_path=gv_cfg["global_vocab_path"],
    global_vocab_meta_path=gv_cfg["global_vocab_meta_path"],
    gene_table_path=gv_cfg["gene_table_path"],
    max_length=int(
        decoder_cfg.get(
            "max_decoder_length",
            data_cfg.get("max_decoder_length", 1025),
        )
    ),
    use_fast=bool(gv_cfg.get("use_fast", True)),
    do_lower_case=bool(gv_cfg.get("do_lower_case", False)),
)

model_cfg = dict(cfg["model"])

for k in [
    "stage1_ckpt_path",
    "freeze_encoder",
    "freeze_embeddings",
    "freeze_value_head",
    "gene_vocab_size",
    "text_vocab_size",
    "vocab_size",
]:
    model_cfg.pop(k, None)

model_cfg["global_vocab_size"] = int(tokenizer.vocab_size)
model_cfg["pad_token_id"] = int(tokenizer.pad_token_id)
model_cfg["mask_value"] = float(data_cfg.get("mask_value", -3.0))
model_cfg["max_decoder_length"] = int(
    decoder_cfg.get(
        "max_decoder_length",
        data_cfg.get("max_decoder_length", 1025),
    )
)

model = ScKITEStage2Model(**model_cfg)

ckpt = torch.load(STAGE2_CKPT, map_location="cpu")
state = (
    ckpt["model_state_dict"]
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt
    else ckpt
)

if state and all(str(k).startswith("module.") for k in state):
    state = {str(k)[7:]: v for k, v in state.items()}

missing, unexpected = model.load_state_dict(state, strict=False)

critical_prefixes = (
    "encoder",
    "shared_token_embedding",
    "value_encoder",
    "annotation_decoder",
    "annotation_lm_head",
)

critical_missing = [
    x for x in missing
    if x.startswith(critical_prefixes)
]

if critical_missing:
    raise RuntimeError(
        "Checkpoint missing critical parameters:\n"
        + "\n".join(critical_missing[:100])
    )

model.to(DEVICE).eval()

for p in model.parameters():
    p.requires_grad_(False)

print("VAL MDS:", PRETRAIN_VAL)
print("all frozen:", all(not p.requires_grad for p in model.parameters()))
print("missing keys:", len(missing))
print("unexpected keys:", len(unexpected))



## 3. 固定 6 类和 external canonical markers

这里使用**预先固定**的 canonical marker list。  
注意：图里的 `NK cell` 对应数据中的 exact label `natural killer cell`。

如果以后要替换 marker reference，应在看 Top-attention gene 结果之前一次性修改并锁定。


In [ ]:





TARGET_CELL_TYPES = {

    "T cell": "T cell",
    "B cell": "B cell",
    "NK cell": "natural killer cell",
    "Plasma cell": "plasma cell",


    "Classical monocyte": "classical monocyte",
    "Macrophage": "macrophage",
    "Neutrophil": "neutrophil",
    "Dendritic cell": "dendritic cell",
    "Mast cell": "mast cell",


    "Endothelial cell": "endothelial cell",
    "Fibroblast": "fibroblast",


    "Epithelial cell": "epithelial cell",


    "Platelet": "platelet",
    "Erythroid cell": "erythroid cell",
}

CANONICAL_MARKERS = {
    "T cell": [
        "CD3D", "CD3E", "CD3G", "TRAC", "CD247",
    ],

    "B cell": [
        "MS4A1", "CD79A", "CD79B", "CD19", "CD22",
    ],

    "NK cell": [
        "GNLY", "KLRD1", "NCR1", "KLRF1", "NKG7", "PRF1",
    ],

    "Plasma cell": [
        "MZB1", "JCHAIN", "XBP1", "PRDM1", "SDC1", "CD38",
    ],

    "Classical monocyte": [
        "CD14", "FCN1", "S100A8", "S100A9", "FCGR1A", "LYZ",
    ],

    "Macrophage": [
        "C1QA", "C1QB", "C1QC", "CD68", "CSF1R", "APOE",
    ],

    "Neutrophil": [
        "FCGR3B", "CSF3R", "CXCR2", "S100A8", "S100A9", "MMP9",
    ],

    "Dendritic cell": [
        "CD1C", "FCER1A", "CLEC10A", "CST3", "HLA-DRA",
    ],

    "Mast cell": [
        "TPSAB1", "KIT", "MS4A2", "CPA3", "CMA1",
    ],

    "Endothelial cell": [
        "PECAM1", "VWF", "CDH5", "CLDN5", "FLT1",
    ],

    "Fibroblast": [
        "DCN", "COL1A1", "COL1A2", "COL3A1", "PDGFRA",
    ],

    "Epithelial cell": [
        "EPCAM", "KRT8", "KRT18", "KRT19", "CDH1",
    ],

    "Platelet": [
        "PPBP", "PF4", "NRGN", "GP9", "TUBB1",
    ],

    "Erythroid cell": [
        "HBB", "HBA1", "HBA2", "ALAS2", "GYPA",
    ],
}
EXACT_TO_FIGURE = {
    exact: display
    for display, exact in TARGET_CELL_TYPES.items()
}

target_exact_labels = set(TARGET_CELL_TYPES.values())

for display_name, exact_name in TARGET_CELL_TYPES.items():
    print(
        f"{display_name:20s} | dataset label = {exact_name:25s} | "
        f"canonical markers = {CANONICAL_MARKERS[display_name]}"
    )


In [ ]:





annotation_decoder_tasks = [
    {
        "name": "annotation",
        "field": ANNOTATION_FIELD,
        "target_type": "plain_text",
        "enabled": True,
    }
]

collator = Stage2Collator(
    tokenizer=tokenizer,
    max_encoder_length=int(data_cfg.get("max_encoder_length", 2049)),
    max_decoder_length=int(
        decoder_cfg.get(
            "max_decoder_length",
            data_cfg.get("max_decoder_length", 1025),
        )
    ),
    mlm_probability=float(ANALYSIS_MLM_PROBABILITY),
    num_bins=int(data_cfg.get("num_bins", 51)),
    sampling=bool(data_cfg.get("sampling", False)),
    keep_first_n_tokens=int(data_cfg.get("keep_first_n_tokens", 1)),
    pad_value=float(data_cfg.get("pad_value", -2.0)),
    cls_value=float(data_cfg.get("cls_value", -1.0)),
    mask_value=float(data_cfg.get("mask_value", -3.0)),
    mask_gene_input=bool(data_cfg.get("mask_gene_input", False)),
    mask_expr_input=bool(data_cfg.get("mask_expr_input", True)),
    decoder_tasks=annotation_decoder_tasks,
    genes_field=str(data_cfg.get("genes_field", GENES_FIELD)),
    expressions_field=str(data_cfg.get("expressions_field", EXPRESSIONS_FIELD)),
    loss_weight_field=data_cfg.get("loss_weight_field", "loss_weight"),
    use_loss_weight=False,
    task_sampling_mode="all",
    regulon_target_path=cfg["paths"].get("regulon_target_path"),
    active_regulon_field=str(
        data_cfg.get("active_regulon_field", "active_regulon_ids")
    ),
    cell_id_field=str(data_cfg.get("cell_id_field", CELL_ID_FIELD)),
    n_regulons=int(data_cfg.get("n_regulons", 530)),
    regulon_num_queries=int(data_cfg.get("regulon_num_queries", 3)),
    regulon_sampling_mode=str(
        data_cfg.get("regulon_sampling_mode", "cyclic_without_replacement")
    ),
    regulon_seed=int(data_cfg.get("regulon_seed", 42)),
    validation_regulon_seed=int(
        data_cfg.get("validation_regulon_seed", 2026)
    ),
    cell_specific_targets=bool(
        data_cfg.get("cell_specific_targets", True)
    ),
    expressed_gene_threshold=float(
        data_cfg.get("expressed_gene_threshold", 0.0)
    ),
    dynamic_target_budget=bool(
        data_cfg.get("dynamic_target_budget", True)
    ),
    is_training=False,
    fixed_eval_encoder_mask=bool(
        data_cfg.get("fixed_eval_encoder_mask", True)
    ),
)

print("analysis mlm_probability:", collator.mlm_probability)
print("mask_gene_input:", collator.mask_gene_input)
print("mask_expr_input:", collator.mask_expr_input)
print("fixed_eval_encoder_mask:", collator.fixed_eval_encoder_mask)

if float(collator.mlm_probability) != 0.0:
    raise RuntimeError("本实验要求 mlm_probability=0.0。")


In [ ]:





val_ds = StreamingDataset(
    local=str(PRETRAIN_VAL),
    shuffle=False,
    batch_size=1,
    allow_unsafe_types=True,
)

print("VAL cells:", len(val_ds))

x0 = val_ds[0]

required = [
    CELL_ID_FIELD,
    CELL_TYPE_FIELD,
    ANNOTATION_FIELD,
    GENES_FIELD,
    EXPRESSIONS_FIELD,
]

missing_fields = [
    x for x in required
    if x not in x0
]

if missing_fields:
    raise KeyError(f"MDS missing fields: {missing_fields}")

print("VAL keys:", sorted(x0.keys()))


def normalize_spaces(text):
    text = "" if text is None else str(text)
    text = text.replace("\u00A0", " ")
    return " ".join(text.split())


def find_subsequence(sequence, subsequence):
    sequence = list(sequence)
    subsequence = list(subsequence)

    if not subsequence or len(subsequence) > len(sequence):
        return []

    n = len(subsequence)

    return [
        i
        for i in range(len(sequence) - n + 1)
        if sequence[i:i+n] == subsequence
    ]


def locate_celltype_span(cell_type, annotation):
    cell_type = normalize_spaces(cell_type)
    annotation = normalize_spaces(annotation)

    if not cell_type or not annotation:
        return {
            "success": False,
            "reason": "empty_celltype_or_annotation",
        }

    m = re.search(
        re.escape(cell_type),
        annotation,
        flags=re.IGNORECASE,
    )

    if m is None:
        return {
            "success": False,
            "reason": "no_exact_substring",
        }

    matched_surface = annotation[m.start():m.end()]

    body_ids = tokenizer.encode_plain_text_to_global_ids(annotation)
    span_ids = tokenizer.encode_plain_text_to_global_ids(matched_surface)

    hits = find_subsequence(body_ids, span_ids)

    if not hits:
        return {
            "success": False,
            "reason": "text_match_but_token_match_failed",
        }

    body_start = int(hits[0])
    body_end = body_start + len(span_ids)

    task_name_ids = tokenizer.encode_task_name_to_global_ids("annotation")
    body_offset = 1 + len(task_name_ids) + 1

    query_positions = list(
        range(
            body_offset + body_start,
            body_offset + body_end,
        )
    )

    io = tokenizer.build_generic_task_decoder_io(
        task_name="annotation",
        content=annotation,
        target_type="plain_text",
        max_length=int(model.max_decoder_length),
    )

    if not query_positions:
        return {
            "success": False,
            "reason": "empty_query_positions",
        }

    if max(query_positions) >= len(io["decoder_input_ids"]):
        return {
            "success": False,
            "reason": "celltype_span_truncated",
        }

    return {
        "success": True,
        "reason": "ok",
        "matched_surface_text": matched_surface,
        "n_celltype_tokens": int(len(span_ids)),
        "n_token_occurrences": int(len(hits)),
        "decoder_query_positions": query_positions,
    }


In [ ]:





span_index_path = OUT_DIR / "selected_celltypes_span_index.csv"

def scan_selected_celltypes():
    rows = []

    for i in tqdm(range(len(val_ds)), desc="scan selected cell types"):
        rec = val_ds[i]

        cell_type = normalize_spaces(
            rec.get(CELL_TYPE_FIELD, "")
        )

        if cell_type not in target_exact_labels:
            continue

        annotation = normalize_spaces(
            rec.get(ANNOTATION_FIELD, "")
        )

        info = locate_celltype_span(
            cell_type,
            annotation,
        )

        rows.append({
            "dataset_index": int(i),
            "cell_id": str(rec.get(CELL_ID_FIELD, i)),
            "cell_type": cell_type,
            "figure_cell_type": EXACT_TO_FIGURE[cell_type],
            "natural_language_annotation": annotation,
            "span_success": bool(info.get("success", False)),
            "span_reason": info.get("reason", ""),
            "matched_surface_text": info.get("matched_surface_text", ""),
            "n_celltype_tokens": int(info.get("n_celltype_tokens", 0)),
        })

    return pd.DataFrame(rows)


if span_index_path.exists() and not FORCE_RESCAN:
    span_df = pd.read_csv(span_index_path)
else:
    span_df = scan_selected_celltypes()
    span_df.to_csv(span_index_path, index=False)

span_summary = (
    span_df
    .groupby("figure_cell_type", as_index=False)
    .agg(
        n_val_cells=("cell_id", "nunique"),
        n_exact_span=("span_success", "sum"),
    )
)

span_summary["exact_span_fraction"] = (
    span_summary["n_exact_span"]
    / span_summary["n_val_cells"]
)

display(span_summary)

eligible = span_df[
    span_df["span_success"]
].copy()

if MAX_CELLS_PER_TYPE is not None:
    eligible = (
        eligible
        .sort_values(
            ["figure_cell_type", "dataset_index"],
            kind="stable",
        )
        .groupby(
            "figure_cell_type",
            group_keys=False,
        )
        .head(int(MAX_CELLS_PER_TYPE))
        .reset_index(drop=True)
    )

print("\nCells entering attention extraction:")
display(
    eligible.groupby(
        "figure_cell_type"
    ).size().rename("n_cells").to_frame()
)


In [ ]:





def gene_name(gid):
    gid = int(gid)

    x = tokenizer.global_id_to_gene_symbol.get(gid)
    if x is not None and str(x).strip():
        return str(x).upper()

    x = tokenizer.global_id_to_ensembl.get(gid)
    if x is not None and str(x).strip():
        return str(x)

    return str(
        tokenizer.global_id_to_token.get(gid, gid)
    )


symbol_to_gene_ids = {}

for gid in tokenizer.global_id_to_token.keys():
    gid = int(gid)

    if not tokenizer.global_id_is_gene(gid):
        continue

    symbol = tokenizer.global_id_to_gene_symbol.get(gid)

    if symbol is None:
        continue

    symbol = str(symbol).strip().upper()

    if not symbol:
        continue

    symbol_to_gene_ids.setdefault(
        symbol,
        []
    ).append(gid)


availability_rows = []

for cell_type, markers in CANONICAL_MARKERS.items():
    for marker in markers:
        marker_u = marker.upper()
        ids = symbol_to_gene_ids.get(marker_u, [])

        availability_rows.append({
            "cell_type": cell_type,
            "canonical_marker": marker_u,
            "in_global_gene_vocab": bool(ids),
            "global_gene_ids": ",".join(map(str, ids)),
        })

marker_availability_df = pd.DataFrame(
    availability_rows
)

marker_availability_df.to_csv(
    OUT_DIR / "canonical_marker_vocab_availability.csv",
    index=False,
)

display(marker_availability_df)

missing_marker_vocab = marker_availability_df[
    ~marker_availability_df["in_global_gene_vocab"]
]

if len(missing_marker_vocab):
    print("\nCanonical markers absent from model gene vocabulary:")
    display(missing_marker_vocab)


In [ ]:





def move_tensor_batch_to_device(cpu_batch):
    return {
        k: (
            v.to(DEVICE, non_blocking=True)
            if torch.is_tensor(v)
            else v
        )
        for k, v in cpu_batch.items()
    }


def autocast_ctx():
    if DEVICE.type == "cuda":
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return nullcontext()


def attention_percentile(values):
    x = np.asarray(
        values,
        dtype=np.float64,
    )

    if len(x) == 0:
        return np.asarray([], dtype=np.float64)

    order = np.argsort(
        x,
        kind="mergesort",
    )

    sorted_x = x[order]
    ranks = np.empty(
        len(x),
        dtype=np.float64,
    )

    start = 0

    while start < len(x):
        end = start + 1

        while (
            end < len(x)
            and sorted_x[end] == sorted_x[start]
        ):
            end += 1

        avg_rank = (
            (start + 1) + end
        ) / 2.0

        ranks[
            order[start:end]
        ] = avg_rank

        start = end

    return (
        ranks - 0.5
    ) / len(x)


def save_gene_part(df, part_no, parts_dir):
    parquet_path = (
        parts_dir
        / f"part_{part_no:06d}.parquet"
    )

    try:
        df.to_parquet(
            parquet_path,
            index=False,
        )
        return parquet_path

    except Exception:
        csv_path = (
            parts_dir
            / f"part_{part_no:06d}.csv.gz"
        )

        df.to_csv(
            csv_path,
            index=False,
            compression="gzip",
        )

        return csv_path


def list_gene_parts(parts_dir):
    return sorted(
        list(parts_dir.glob("part_*.parquet"))
        + list(parts_dir.glob("part_*.csv.gz"))
    )


def read_gene_part(path):
    path = Path(path)

    if path.suffix == ".parquet":
        return pd.read_parquet(path)

    return pd.read_csv(
        path,
        compression="gzip",
    )


In [ ]:





parts_dir = OUT_DIR / "selected_celltypes_gene_attention_parts"
parts_dir.mkdir(parents=True, exist_ok=True)

done_path = OUT_DIR / "attention_extraction_done.json"

existing_parts = list_gene_parts(parts_dir)

if (
    done_path.exists()
    and existing_parts
    and not FORCE_RERUN_ATTENTION
):
    print(
        f"Loaded cached gene-level attention parts: "
        f"{len(existing_parts)}"
    )

else:
    for p in existing_parts:
        p.unlink()

    records_meta = eligible[
        [
            "dataset_index",
            "cell_id",
            "cell_type",
            "figure_cell_type",
        ]
    ].to_records(index=False)

    part_no = 0
    n_cells_written = 0

    for start in tqdm(
        range(
            0,
            len(records_meta),
            ATTENTION_BATCH_SIZE,
        ),
        desc="selected-cell cross-attention",
    ):
        chunk = records_meta[
            start:start + ATTENTION_BATCH_SIZE
        ]

        records = [
            val_ds[int(x.dataset_index)]
            for x in chunk
        ]

        span_infos = [
            locate_celltype_span(
                rec[CELL_TYPE_FIELD],
                rec[ANNOTATION_FIELD],
            )
            for rec in records
        ]

        cpu_batch = collator(
            [dict(x) for x in records]
        )

        if len(cpu_batch["cell_keys"]) != len(records):
            raise RuntimeError(
                "Annotation-only collator 输出行数与输入 records 不一致。"
            )

        batch = move_tensor_batch_to_device(
            cpu_batch
        )

        with torch.inference_mode():
            with autocast_ctx():
                outputs = model(
                    encoder_input_gene_ids=batch[
                        "encoder_input_gene_ids"
                    ],
                    encoder_input_values=batch[
                        "encoder_input_values"
                    ],
                    encoder_key_padding_mask=batch[
                        "encoder_key_padding_mask"
                    ],
                    decoder_input_ids=batch[
                        "decoder_input_ids"
                    ],
                    decoder_attention_mask=batch[
                        "decoder_attention_mask"
                    ],
                    decoder_task_ids=batch[
                        "decoder_task_ids"
                    ],
                    return_encoder_gene_logits=False,
                    return_last_cross_attn=True,
                )

        cross = outputs[
            "annotation_last_cross_attn"
        ].detach().float().cpu()

        if cross.ndim != 4:
            raise RuntimeError(
                "annotation_last_cross_attn shape 异常: "
                f"{tuple(cross.shape)}"
            )

        target_gene_ids = cpu_batch[
            "encoder_target_gene_ids"
        ].cpu()

        encoder_kpm = cpu_batch[
            "encoder_key_padding_mask"
        ].cpu()

        batch_rows = []

        for b, rec in enumerate(records):
            sinfo = span_infos[b]

            if not sinfo.get("success", False):
                continue

            qpos = torch.tensor(
                sinfo["decoder_query_positions"],
                dtype=torch.long,
            )

            if int(qpos.max()) >= cross.shape[2]:
                continue

            gene_att = (
                cross[b, :, qpos, :]
                .mean(dim=(0, 1))
                .numpy()
            )

            gids = target_gene_ids[b].numpy()
            pads = encoder_kpm[b].numpy().astype(bool)

            valid_gid = []
            valid_att = []

            for gid, att, is_pad in zip(
                gids,
                gene_att,
                pads,
            ):
                gid = int(gid)

                if is_pad:
                    continue

                if gid == int(tokenizer.cls_token_id):
                    continue

                if not tokenizer.global_id_is_gene(gid):
                    continue

                valid_gid.append(gid)
                valid_att.append(float(att))

            if not valid_gid:
                continue

            pct = attention_percentile(
                valid_att
            )

            exact_cell_type = normalize_spaces(
                rec[CELL_TYPE_FIELD]
            )

            figure_cell_type = EXACT_TO_FIGURE[
                exact_cell_type
            ]

            canonical_set = {
                x.upper()
                for x in CANONICAL_MARKERS[
                    figure_cell_type
                ]
            }

            cell_id = str(
                rec[CELL_ID_FIELD]
            )

            for gid, raw_att, p in zip(
                valid_gid,
                valid_att,
                pct,
            ):
                symbol = gene_name(gid)

                batch_rows.append({
                    "cell_id": cell_id,
                    "cell_type": exact_cell_type,
                    "figure_cell_type": figure_cell_type,
                    "gene_token_id": int(gid),
                    "gene": symbol,
                    "attention_raw": float(raw_att),
                    "attention_percentile": float(p),
                    "is_canonical_marker": bool(
                        symbol.upper()
                        in canonical_set
                    ),
                })

            n_cells_written += 1

        if batch_rows:
            save_gene_part(
                pd.DataFrame(batch_rows),
                part_no,
                parts_dir,
            )
            part_no += 1

    with open(
        done_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "n_selected_cells": int(len(eligible)),
                "n_cells_written": int(n_cells_written),
                "n_parts": int(part_no),
                "attention_batch_size": int(ATTENTION_BATCH_SIZE),
                "mlm_probability": float(
                    ANALYSIS_MLM_PROBABILITY
                ),
            },
            f,
            indent=2,
        )

    print(
        "attention extraction finished:",
        n_cells_written,
        "cells",
    )

print("parts:", len(list_gene_parts(parts_dir)))


In [ ]:





parts = list_gene_parts(parts_dir)

if not parts:
    raise RuntimeError(
        "没有找到 gene-level attention parts。"
    )

pieces = []

for p in tqdm(
    parts,
    desc="read attention parts",
):
    pieces.append(
        read_gene_part(p)
    )

gene_attention_df = pd.concat(
    pieces,
    ignore_index=True,
)

n_cells_by_type = (
    gene_attention_df
    .groupby("figure_cell_type")[
        "cell_id"
    ]
    .nunique()
    .to_dict()
)

gene_summary = (
    gene_attention_df
    .groupby(
        [
            "figure_cell_type",
            "gene_token_id",
            "gene",
        ],
        as_index=False,
    )
    .agg(
        median_attention_percentile=(
            "attention_percentile",
            "median",
        ),
        mean_attention_percentile=(
            "attention_percentile",
            "mean",
        ),
        median_attention_raw=(
            "attention_raw",
            "median",
        ),
        n_cells_present=(
            "cell_id",
            "nunique",
        ),
    )
)

gene_summary["n_analyzed_cells"] = (
    gene_summary[
        "figure_cell_type"
    ].map(n_cells_by_type)
)

gene_summary["presence_fraction"] = (
    gene_summary["n_cells_present"]
    / gene_summary["n_analyzed_cells"]
)

canonical_map = {
    ct: {
        x.upper()
        for x in markers
    }
    for ct, markers
    in CANONICAL_MARKERS.items()
}

gene_summary["is_canonical_marker"] = [
    str(gene).upper()
    in canonical_map[cell_type]
    for cell_type, gene
    in zip(
        gene_summary["figure_cell_type"],
        gene_summary["gene"],
    )
]

gene_summary["passes_presence_filter"] = (
    (
        gene_summary["n_cells_present"]
        >= int(MIN_GENE_CELLS)
    )
    &
    (
        gene_summary["presence_fraction"]
        >= float(
            MIN_GENE_PRESENCE_FRACTION
        )
    )
)

eligible_gene_summary = gene_summary[
    gene_summary["passes_presence_filter"]
].copy()

eligible_gene_summary = (
    eligible_gene_summary
    .sort_values(
        [
            "figure_cell_type",
            "median_attention_percentile",
            "presence_fraction",
            "mean_attention_percentile",
            "gene",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            True,
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

eligible_gene_summary["attention_rank"] = (
    eligible_gene_summary
    .groupby("figure_cell_type")
    .cumcount()
    + 1
)

gene_summary.to_csv(
    OUT_DIR / "all_gene_attention_summary.csv.gz",
    index=False,
    compression="gzip",
)

eligible_gene_summary.to_csv(
    OUT_DIR / "eligible_gene_attention_ranking.csv",
    index=False,
)

display(
    eligible_gene_summary.head(30)
)


In [ ]:





top_genes_df = (
    eligible_gene_summary[
        eligible_gene_summary[
            "attention_rank"
        ] <= int(TOP_N_GENES)
    ]
    .copy()
)

top_genes_df.to_csv(
    OUT_DIR / f"top{TOP_N_GENES}_attention_genes_by_celltype.csv",
    index=False,
)


canonical_rank_rows = []

for cell_type, markers in CANONICAL_MARKERS.items():
    sub_all = gene_summary[
        gene_summary[
            "figure_cell_type"
        ] == cell_type
    ]

    sub_rank = eligible_gene_summary[
        eligible_gene_summary[
            "figure_cell_type"
        ] == cell_type
    ]

    rank_lookup = {
        str(r.gene).upper(): r
        for r in sub_rank.itertuples(
            index=False
        )
    }

    raw_lookup = {
        str(r.gene).upper(): r
        for r in sub_all.itertuples(
            index=False
        )
    }

    for marker in markers:
        marker_u = marker.upper()

        rr = rank_lookup.get(marker_u)
        raw = raw_lookup.get(marker_u)

        canonical_rank_rows.append({
            "cell_type": cell_type,
            "canonical_marker": marker_u,
            "observed_in_encoder_cells": raw is not None,
            "passes_presence_filter": rr is not None,
            "attention_rank": (
                int(rr.attention_rank)
                if rr is not None
                else np.nan
            ),
            "median_attention_percentile": (
                float(
                    raw.median_attention_percentile
                )
                if raw is not None
                else np.nan
            ),
            "presence_fraction": (
                float(raw.presence_fraction)
                if raw is not None
                else np.nan
            ),
            "n_cells_present": (
                int(raw.n_cells_present)
                if raw is not None
                else 0
            ),
        })

canonical_rank_df = pd.DataFrame(
    canonical_rank_rows
)

canonical_rank_df.to_csv(
    OUT_DIR / "canonical_marker_attention_ranks.csv",
    index=False,
)


overlap_rows = []

for cell_type in TARGET_CELL_TYPES.keys():
    ranked = eligible_gene_summary[
        eligible_gene_summary[
            "figure_cell_type"
        ] == cell_type
    ].sort_values(
        "attention_rank"
    )

    topk = ranked.head(
        int(TOP_K_FOR_OVERLAP)
    )

    canonical_set = canonical_map[
        cell_type
    ]

    hit_genes = [
        g
        for g in topk["gene"].astype(str)
        if g.upper() in canonical_set
    ]

    canonical_ranks = canonical_rank_df[
        (
            canonical_rank_df["cell_type"]
            == cell_type
        )
        &
        (
            canonical_rank_df[
                "attention_rank"
            ].notna()
        )
    ]["attention_rank"].astype(float)

    overlap_rows.append({
        "cell_type": cell_type,
        "n_analyzed_cells": int(
            n_cells_by_type.get(
                cell_type,
                0,
            )
        ),
        "top_k": int(
            TOP_K_FOR_OVERLAP
        ),
        "n_canonical_markers_fixed": int(
            len(canonical_set)
        ),
        "n_canonical_hits_in_top_k": int(
            len(hit_genes)
        ),
        "canonical_hits_in_top_k": ",".join(
            hit_genes
        ),
        "top_k_hit_fraction": float(
            len(hit_genes)
            / int(TOP_K_FOR_OVERLAP)
        ),
        "median_canonical_marker_rank": (
            float(
                np.median(
                    canonical_ranks
                )
            )
            if len(canonical_ranks)
            else np.nan
        ),
    })

overlap_df = pd.DataFrame(
    overlap_rows
)

overlap_df.to_csv(
    OUT_DIR / f"top{TOP_K_FOR_OVERLAP}_canonical_marker_overlap_summary.csv",
    index=False,
)

print("=== Top genes ===")
display(top_genes_df)

print("=== Canonical marker ranks ===")
display(canonical_rank_df)

print("=== Overlap summary ===")
display(overlap_df)


In [ ]:





for cell_type in TARGET_CELL_TYPES.keys():
    print("\n" + "=" * 100)
    print(cell_type)
    print("=" * 100)

    sub = (
        top_genes_df[
            top_genes_df[
                "figure_cell_type"
            ] == cell_type
        ]
        .sort_values(
            "attention_rank"
        )
        [
            [
                "attention_rank",
                "gene",
                "median_attention_percentile",
                "presence_fraction",
                "n_cells_present",
                "is_canonical_marker",
            ]
        ]
    )

    display(sub)


In [ ]:





CANONICAL_COLOR = "#6FAF45"
OTHER_COLOR = "#B7B7B7"

fig, axes = plt.subplots(
    2,
    3,
    figsize=(8.0, 6.0),
)

axes = axes.flatten()

cell_type_order = [
    "T cell",
    "B cell",
    "NK cell",
    "Classical monocyte",
    "Plasma cell",
    "Neutrophil",
]

for ax, cell_type in zip(
    axes,
    cell_type_order,
):
    sub = (
        top_genes_df[
            top_genes_df[
                "figure_cell_type"
            ] == cell_type
        ]
        .sort_values(
            "attention_rank",
            ascending=False,
        )
        .copy()
    )

    if len(sub) == 0:
        ax.text(
            0.5,
            0.5,
            "No eligible genes",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=TICK_SIZE,
        )
        ax.set_title(
            cell_type,
            fontsize=TITLE_SIZE,
            fontweight="bold",
        )
        ax.axis("off")
        continue

    colors = [
        (
            CANONICAL_COLOR
            if bool(x)
            else OTHER_COLOR
        )
        for x in sub[
            "is_canonical_marker"
        ]
    ]

    y = np.arange(
        len(sub)
    )

    ax.barh(
        y,
        sub[
            "median_attention_percentile"
        ],
        color=colors,
        edgecolor="black",
        linewidth=0.35,
        height=0.72,
    )

    ax.set_yticks(y)
    ax.set_yticklabels(
        sub["gene"],
        fontsize=TICK_SIZE,
    )

    for tick, is_marker in zip(
        ax.get_yticklabels(),
        sub["is_canonical_marker"],
    ):
        if bool(is_marker):
            tick.set_fontweight(
                "bold"
            )
            tick.set_color(
                CANONICAL_COLOR
            )

    ax.set_xlim(
        0,
        1.0,
    )

    ax.set_title(
        cell_type,
        fontsize=TITLE_SIZE,
        fontweight="bold",
        pad=4,
    )

    ax.tick_params(
        axis="x",
        labelsize=TICK_SIZE,
        width=0.7,
        length=3,
    )

    ax.tick_params(
        axis="y",
        width=0,
        length=0,
    )

    ax.spines["top"].set_visible(
        False
    )
    ax.spines["right"].set_visible(
        False
    )

    ax.spines["left"].set_linewidth(
        0.7
    )
    ax.spines["bottom"].set_linewidth(
        0.7
    )

fig.supxlabel(
    "Median cell-type-span cross-attention percentile",
    fontsize=LABEL_SIZE,
    y=0.02,
)

fig.text(
    0.5,
    0.995,
    f"Top {TOP_N_GENES} attention-ranked genes in common immune cell types",
    ha="center",
    va="top",
    fontsize=TITLE_SIZE,
    fontweight="bold",
)

fig.tight_layout(
    rect=[
        0.02,
        0.05,
        1,
        0.96,
    ]
)

for ext in [
    "pdf",
    "svg",
    "png",
]:
    kwargs = {
        "bbox_inches": "tight",
    }

    if ext == "png":
        kwargs["dpi"] = 600

    fig.savefig(
        OUT_DIR
        / f"fig5c_top{TOP_N_GENES}_attention_genes_canonical_markers.{ext}",
        **kwargs,
    )

plt.show()


In [ ]:





plot_df = canonical_rank_df[
    canonical_rank_df[
        "attention_rank"
    ].notna()
].copy()

if len(plot_df):
    fig, axes = plt.subplots(
        2,
        3,
        figsize=(8.0, 5.3),
    )

    axes = axes.flatten()

    for ax, cell_type in zip(
        axes,
        cell_type_order,
    ):
        sub = (
            plot_df[
                plot_df[
                    "cell_type"
                ] == cell_type
            ]
            .sort_values(
                "attention_rank",
                ascending=False,
            )
        )

        if len(sub) == 0:
            ax.axis("off")
            continue

        y = np.arange(
            len(sub)
        )

        ax.barh(
            y,
            sub["attention_rank"],
            color=CANONICAL_COLOR,
            edgecolor="black",
            linewidth=0.35,
        )

        ax.set_yticks(y)

        ax.set_yticklabels(
            sub["canonical_marker"],
            fontsize=TICK_SIZE,
            fontweight="bold",
        )

        ax.set_title(
            cell_type,
            fontsize=TITLE_SIZE,
            fontweight="bold",
        )

        ax.tick_params(
            axis="x",
            labelsize=TICK_SIZE,
        )

        ax.spines["top"].set_visible(
            False
        )
        ax.spines["right"].set_visible(
            False
        )

    fig.supxlabel(
        "Attention rank among robustly present encoder genes (smaller = higher)",
        fontsize=LABEL_SIZE,
        y=0.02,
    )

    fig.tight_layout(
        rect=[
            0.02,
            0.05,
            1,
            1,
        ]
    )

    for ext in [
        "pdf",
        "svg",
        "png",
    ]:
        kwargs = {
            "bbox_inches": "tight",
        }

        if ext == "png":
            kwargs["dpi"] = 600

        fig.savefig(
            OUT_DIR
            / f"fig5c_canonical_marker_attention_ranks.{ext}",
            **kwargs,
        )

    plt.show()


In [ ]:





files = sorted(
    x
    for x in OUT_DIR.rglob("*")
    if x.is_file()
)

inventory = pd.DataFrame({
    "file": [
        str(
            x.relative_to(
                OUT_DIR
            )
        )
        for x in files
    ],
    "size_MB": [
        round(
            x.stat().st_size
            / 1024**2,
            3,
        )
        for x in files
    ],
})

display(inventory)


In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D





CSV_PATH = OUT_DIR / 'top15_attention_genes_by_celltype.csv'
OUT_PREFIX = "b_cell_top8_table_dotplot"

TARGET_CELL_TYPE = "B cell"
TOP_N = 8

CANONICAL_MARKERS = {
    "MS4A1", "CD79A", "CD79B", "CD19", "PAX5"
}

ARIAL_TTF = None


TITLE_SIZE = 10
HEADER_SIZE = 8
TEXT_SIZE = 8
SMALL_SIZE = 7


MARKER_COLOR = "#8CBF45"
MARKER_TEXT = "#568522"

OTHER_COLOR = "#BDBDBD"
OTHER_TEXT = "#333333"

ROW_MARKER_BG = "#F0F6E7"
GRID_COLOR = "#E6E6E6"
AXIS_COLOR = "#555555"





def configure_arial(arial_ttf=None):
    if arial_ttf is not None:
        fm.fontManager.addfont(arial_ttf)

    try:
        arial_path = fm.findfont("Arial", fallback_to_default=False)
        print("Arial found:", arial_path)
        family = "Arial"
    except Exception:
        print("Arial not found; using sans-serif.")
        family = "sans-serif"

    mpl.rcParams.update({
        "font.family": family,
        "axes.unicode_minus": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })

configure_arial(ARIAL_TTF)





df = pd.read_csv(CSV_PATH)

required = [
    "figure_cell_type",
    "attention_rank",
    "gene",
    "median_attention_percentile",
    "presence_fraction",
    "n_cells_present",
    "is_canonical_marker",
]

missing = [x for x in required if x not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

sub = (
    df[df["figure_cell_type"] == TARGET_CELL_TYPE]
    .sort_values("attention_rank")
    .head(TOP_N)
    .copy()
    .reset_index(drop=True)
)

if len(sub) == 0:
    raise ValueError(f"No data found for {TARGET_CELL_TYPE}")

sub["is_canonical_marker"] = [
    bool(flag) or str(gene).upper() in CANONICAL_MARKERS
    for flag, gene in zip(sub["is_canonical_marker"], sub["gene"])
]

print(sub[[
    "attention_rank", "gene",
    "median_attention_percentile",
    "is_canonical_marker"
]])





n = len(sub)
y = np.arange(n)[::-1]

attention = sub["median_attention_percentile"].astype(float).to_numpy()
is_marker = sub["is_canonical_marker"].astype(bool).to_numpy()





fig = plt.figure(figsize=(5.4, 3.8))


ax_left = fig.add_axes([0.06, 0.18, 0.34, 0.66])


ax_right = fig.add_axes([0.40, 0.18, 0.54, 0.66])





ax_left.set_xlim(0, 1)
ax_left.set_ylim(-0.7, n - 0.3)
ax_left.axis("off")


fig.text(
    0.06, 0.95,
    "B cell",
    ha="left", va="center",
    fontsize=TITLE_SIZE,
    fontweight="bold"
)


header_y = n - 0.25

ax_left.text(
    0.08, header_y,
    "Rank",
    ha="center", va="bottom",
    fontsize=HEADER_SIZE,
    fontweight="bold"
)

ax_left.text(
    0.34, header_y,
    "Gene",
    ha="left", va="bottom",
    fontsize=HEADER_SIZE,
    fontweight="bold"
)

ax_left.plot(
    [0, 1],
    [n - 0.38, n - 0.38],
    color=AXIS_COLOR,
    linewidth=0.8,
)


for i, row in sub.iterrows():
    yi = y[i]
    canonical = bool(row["is_canonical_marker"])

    if canonical:
        ax_left.axhspan(
            yi - 0.43,
            yi + 0.43,
            xmin=0,
            xmax=1,
            facecolor=ROW_MARKER_BG,
            edgecolor="none",
            zorder=0,
        )

    ax_left.text(
        0.08,
        yi,
        str(int(row["attention_rank"])),
        ha="center",
        va="center",
        fontsize=TEXT_SIZE,
        color="#666666",
    )

    ax_left.text(
        0.34,
        yi,
        str(row["gene"]),
        ha="left",
        va="center",
        fontsize=TEXT_SIZE,
        color=MARKER_TEXT if canonical else OTHER_TEXT,
        fontweight="bold" if canonical else "normal",
    )





xmin = max(0, np.floor((attention.min() - 0.015) * 100) / 100)
xmax = min(1.0, np.ceil((attention.max() + 0.005) * 100) / 100)

if xmax - xmin < 0.05:
    xmin = max(0, xmax - 0.06)

ax_right.set_xlim(xmin, xmax)
ax_right.set_ylim(-0.7, n - 0.3)


for yi in y:
    ax_right.hlines(
        yi,
        xmin=xmin,
        xmax=xmax,
        color=GRID_COLOR,
        linewidth=0.7,
        zorder=0,
    )


for yi, value, canonical in zip(y, attention, is_marker):
    ax_right.hlines(
        yi,
        xmin=xmin,
        xmax=value,
        color=MARKER_COLOR if canonical else OTHER_COLOR,
        linewidth=1.5,
        alpha=0.75,
        zorder=1,
    )


dot_colors = [MARKER_COLOR if x else OTHER_COLOR for x in is_marker]
ax_right.scatter(
    attention,
    y,
    s=60,
    c=dot_colors,
    edgecolor="black",
    linewidth=0.5,
    zorder=3,
)






xrange_ = xmax - xmin

for yi, value, canonical in zip(y, attention, is_marker):

    x_text = value + 0.018 * xrange_


    x_text = min(x_text, xmax - 0.004)

    ax_right.text(
        x_text,
        yi,
        f"{value:.3f}",
        ha="left",
        va="center",
        fontsize=SMALL_SIZE,
        color=MARKER_TEXT if canonical else "#666666",
        fontweight="bold" if canonical else "normal",
    )





ax_right.set_yticks([])

ax_right.set_xlabel(
    "Median cell-type-span cross-attention percentile",
    fontsize=HEADER_SIZE,
    labelpad=6,
)

ax_right.tick_params(
    axis="x",
    labelsize=SMALL_SIZE,
    width=0.7,
    length=3,
)

ax_right.spines["top"].set_visible(False)
ax_right.spines["right"].set_visible(False)
ax_right.spines["left"].set_visible(False)

ax_right.spines["bottom"].set_linewidth(0.8)
ax_right.spines["bottom"].set_color(AXIS_COLOR)





legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor=MARKER_COLOR,
        markeredgecolor="none",
        markersize=6,
        label="Canonical B-cell marker",
    ),
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor=OTHER_COLOR,
        markeredgecolor="none",
        markersize=6,
        label="Other top-attended gene",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower left",
    bbox_to_anchor=(0.06, 0.025),
    ncol=2,
    frameon=False,
    fontsize=SMALL_SIZE,
    handlelength=0.8,
    handletextpad=0.5,
    columnspacing=1.8,
)





for ext in ["pdf", "svg", "png"]:
    kwargs = {
        "bbox_inches": "tight",
        "facecolor": "white",
    }
    if ext == "png":
        kwargs["dpi"] = 600

    fig.savefig(f"{OUT_PREFIX}.{ext}", **kwargs)

plt.show()
